<a href="https://colab.research.google.com/github/Shivam-vachhani/PyTorch-Projects/blob/main/CNN_on_Fashion_MNIST_Dataset_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [ ]:
device  =  torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
df = pd.read_csv('fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
class MyCustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
    self.labels = torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [ ]:
train_dataset = MyCustomDataset(X_train,y_train)
test_dataset = MyCustomDataset(X_test,y_test)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True,pin_memory=True)

In [ ]:
# Making CNN by seprate task first feature extraction by CNN and second is classification by ANN

class MyNN(nn.Module):

  def __init__(self,num_channels):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(num_channels,32,kernel_size=3,padding="same"),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(32,64,kernel_size=3,padding="same"),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),
    )

    self.classification = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7,128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,10),
    )


  def forward(self,x):
    x = self.features(x)
    x = self.classification(x)
    return x


In [ ]:
epoch = 100
learning_rate = 0.01

In [ ]:
# Initialize Model
model = MyNN(1)
model.to(device)

# loss function
criterion = nn.CrossEntropyLoss()

# Optimizer function
optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)


In [ ]:
for i in range(epoch):
  total_loss = 0
  for batch_features,batch_labels in train_loader:

    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    #forward pass
    output = model(batch_features)

    #loss calulate
    loss = criterion(output,batch_labels)

    #backward pass
    optimizer.zero_grad()
    loss.backward()

    #update parameters
    optimizer.step()

    total_loss += loss.item()
  print(f"Epoch {i+1} Loss: {total_loss/len(train_loader)}")

Epoch 1 Loss: 0.7790367222428322
Epoch 2 Loss: 0.46942238972584405
Epoch 3 Loss: 0.398097693224748
Epoch 4 Loss: 0.357582596535484
Epoch 5 Loss: 0.3333201817497611
Epoch 6 Loss: 0.3090309706181288
Epoch 7 Loss: 0.2957002267713348
Epoch 8 Loss: 0.27870524165903526
Epoch 9 Loss: 0.26918970291813216
Epoch 10 Loss: 0.2573041550020377
Epoch 11 Loss: 0.24588720383060475
Epoch 12 Loss: 0.23796295160303513
Epoch 13 Loss: 0.2308679993165036
Epoch 14 Loss: 0.21756104637061557
Epoch 15 Loss: 0.2113876923918724
Epoch 16 Loss: 0.20556674977888664
Epoch 17 Loss: 0.19998044011369348
Epoch 18 Loss: 0.195550833756725
Epoch 19 Loss: 0.18307786163066825
Epoch 20 Loss: 0.1815431341044605
Epoch 21 Loss: 0.17570060187826553
Epoch 22 Loss: 0.17123112913314253
Epoch 23 Loss: 0.16750515626370907
Epoch 24 Loss: 0.15437797219430408
Epoch 25 Loss: 0.15339981275595105
Epoch 26 Loss: 0.14898540717673797
Epoch 27 Loss: 0.14905508541408927
Epoch 28 Loss: 0.14203818986720096
Epoch 29 Loss: 0.13844195457563424
Epoch 30

In [ ]:
# Calculating accuracy

model.eval()

total = 0
correct = 0
with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    output = model(batch_features)
    _,predicted = torch.max(output,1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f"Accuracy: {correct/total}")



Accuracy: 0.9215833333333333
